<a href="https://colab.research.google.com/github/robertkigobe/llm_engineering/blob/main/cds_pipelines_demo_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Hardware specifications

In [ ]:
# Let's check the GPU - it should be a Tesla T4

gpu_info = !nvidia-smi
gpu_info = '\n'.join(gpu_info)
if gpu_info.find('failed') >= 0:
  print('Not connected to a GPU')
else:
  print(gpu_info)
  if gpu_info.find('Tesla T4') >= 0:
    print("Success - Connected to a T4")
  else:
    print("NOT CONNECTED TO A T4")

Fri Mar 20 15:06:02 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   47C    P0             27W /   70W |     475MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

# Setup & installs

In [ ]:

!pip -q install -U \
  pandas \
  gradio \
  transformers \
  accelerate \
  sentencepiece


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 38.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.9/42.9 MB 16.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.5/58.5 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 51.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.1 which is incompatible.
bqplot 0.12.45 requires pandas<3.0.0,>=1.0.0, but you have pandas 3.0.1 which is incompatible.
dask-cudf-cu12 26.2.1 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.1 which is incompatible.
cudf-cu12 26.2.1 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.1 which is incompatible.
db-dtypes 1.5.0 requires pandas<3.0.0,>=1.5.3, b

# Imports & global

In [ ]:


import pandas as pd
import gradio as gr
from datetime import datetime

# LLM dependencies (we'll initialize the model in a later cell)
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch




# Load your log data

In [ ]:
import pandas as pdimport# ----------------------------
uploaded_df = None

# ----------------------------
# Schema normalization (fixes missing log_level/error_type)
# ----------------------------
def normalize_schema(df: pd.DataFrame) -> pd.DataFrame:
    COLUMN_ALIASES = {
        # log level
        "level": "log_level",
        "severity": "log_level",
        "loglevel": "log_level",

        # error type
        "exception": "error_type",
        "error": "error_type",
        "errorType": "error_type",
        "err_type": "error_type",

        # message
        "msg": "message",
        "log_message": "message",

        # service
        "app": "service",
        "application": "service",
        "service_name": "service",
    }

    df = df.copy()

    # Copy aliased columns into canonical columns if needed
    for src, dst in COLUMN_ALIASES.items():
        if src in df.columns and dst not in df.columns:
            df[dst] = df[src]

    # Normalize level values
    if "log_level" in df.columns:
        df["log_level"] = df["log_level"].astype(str).str.upper()

    return df

# ----------------------------
# File loader (CSV / JSONL / XLSX)
# ----------------------------
def load_logs_gradio(file):
    global uploaded_df

    if file is None:
        uploaded_df = None
        return pd.DataFrame({"message": ["No file uploaded yet."]})

    path = file.name

    if path.endswith(".csv"):
        df = pd.read_csv(path)
    elif path.endswith(".jsonl") or path.endswith(".ndjson"):
        df = pd.read_json(path, lines=True)
    elif path.endswith(".xlsx"):
        df = pd.read_excel(path)
    else:
        uploaded_df = None
        return pd.DataFrame({"error": [f"Unsupported file type: {path}"]})

    # Timestamp normalization (support @timestamp or timestamp)
    if "@timestamp" in df.columns:
        df["@timestamp"] = pd.to_datetime(df["@timestamp"], errors="coerce", utc=True)
    elif "timestamp" in df.columns:
        df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce", utc=True)

    # ✅ Normalize schema once here
    uploaded_df = normalize_schema(df)

    return uploaded_df.head(15)

# ----------------------------
# Dataset profile (used by UI)
# ----------------------------
def dataset_profile():
    global uploaded_df
    if uploaded_df is None or uploaded_df.empty:
        return "No dataset loaded."
    return (
        f"Loaded ✅ Rows: {len(uploaded_df):,} | Columns: {len(uploaded_df.columns)}\n\n"
        "Columns:\n- " + "\n- ".join(map(str, uploaded_df.columns))
    )


# ----------------------------
# Single source of truth for the dataset


# Normalize & enrich logs (truth layer)

In [ ]:
def enrich_logs(df: pd.DataFrame) -> pd.DataFrame:
    """
    Optional enrichment layer.
    Adds derived columns if source fields exist.
    Safe to call multiple times.
    """
    df = df.copy()

    # Timestamp-based enrichment
    if "@timestamp" in df.columns:
        ts = df["@timestamp"]
    elif "timestamp" in df.columns:
        ts = df["timestamp"]
    else:
        ts = None

    if ts is not None:
        df["date"] = ts.dt.date
        df["hour"] = ts.dt.floor("H")

    # HTTP-style status code bucketing
    if "statusCode" in df.columns:
        df["statusClass"] = (df["statusCode"] // 100).astype("Int64").astype(str) + "xx"

    return df


# Analyst‑safe query functions

In [ ]:
# ================================AB CELL 5
# Analyst-safe query functions
# ================================

import pandas as pd

def ensure_data_loaded():
    if uploaded_df is None or uploaded_df.empty:
        return False, "❌ No data loaded yet. Please upload a log file first."
    return True, None


def normalize_timestamp(df: pd.DataFrame):
    df = df.copy()

    if "@timestamp" in df.columns:
        df["_ts"] = pd.to_datetime(df["@timestamp"], errors="coerce", utc=True)
    elif "timestamp" in df.columns:
        df["_ts"] = pd.to_datetime(df["timestamp"], errors="coerce", utc=True)
    else:
        df["_ts"] = pd.NaT

    return df


def get_error_summary():
    ok, msg = ensure_data_loaded()
    if not ok:
        return msg

    df = normalize_timestamp(uploaded_df)

    return (
        df[df["log_level"] == "ERROR"]
        .groupby("error_type")
        .size()
        .reset_index(name="error_count")
        .sort_values("error_count", ascending=False)
    )


def get_errors_by_service(service_name):
    ok, msg = ensure_data_loaded()
    if not ok:
        return msg

    df = normalize_timestamp(uploaded_df)

    filtered = df[
        (df["log_level"] == "ERROR") &
        (df["service"] == service_name)
    ]

    if filtered.empty:
        return f"✅ No errors found for service: {service_name}"

    return (
        filtered[["_ts", "service", "error_type", "message", "sourceId"]]
        .rename(columns={"_ts": "timestamp"})
        .sort_values("timestamp", ascending=False)
        .head(50)
    )


def get_errors_in_date_range(start_date, end_date):
    ok, msg = ensure_data_loaded()
    if not ok:
        return msg

    df = normalize_timestamp(uploaded_df)

    start_dt = pd.to_datetime(start_date, errors="coerce", utc=True)
    end_dt = pd.to_datetime(end_date, errors="coerce", utc=True)

    mask = (
        (df["_ts"] >= start_dt) &
        (df["_ts"] <= end_dt) &
        (df["log_level"] == "ERROR")
    )

    result = df.loc[mask]

    if result.empty:
        return "✅ No errors found in the selected date range."

    return (
        result[["_ts", "service", "error_type", "message"]]
        .rename(columns={"_ts": "timestamp"})
        .sort_values("timestamp", ascending=False)
        .head(100)
    )


def get_top_errors(limit=5):
    ok, msg = ensure_data_loaded()
    if not ok:
        return msg

    return (
        uploaded_df[uploaded_df["log_level"] == "ERROR"]
        .groupby(["error_type", "service"])
        .size()
        .reset_index(name="count")
        .sort_values("count", ascending=False)
        .head(int(limit))
    )


def explain_error_type(error_type):
    explanations = {
        "TimeoutException": "The service took too long to respond.",
        "NullPointerException": "The system tried to use missing data.",
        "DatabaseConnectionError": "The service could not reach the database.",
        "AuthFailure": "Authentication or authorization failed.",
        "ServiceUnavailable": "The service was temporarily down.",
    }

    return explanations.get(
        error_type,
        "No predefined explanation available for this error type."
    )


# Load Hugging Face model (small & demo‑friendly)

In [ ]:
# ================================ install -U transformers accelerate sentencepiece

import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

MODEL_NAME = "google/flan-t5-small"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)  # <-- do NOT flip tie_word_embeddings

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
model.eval()

def llm_invoke(prompt: str, max_new_tokens: int = 256) -> str:
    """Small, reliable 'invoke' for FLAN-T5 without pipelines or LangChain."""
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True).to(device)
    with torch.no_grad():
        out_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,          # deterministic for demo
            num_beams=2,              # a little better quality than greedy, still fast
            no_repeat_ngram_size=3
        )
    return tokenizer.decode(out_ids[0], skip_special_tokens=True)

# Smoke test
print(llm_invoke("Summarize: Service A has TimeoutException spikes after a deployment.", max_new_tokens=80))

# COLAB CELL 6 — FLAN-T5 (version-proof, no HF pipeline registry)
# ================================



config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/308M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Service A has a timeoutException spike after a deployment.


# LLM prompt for analyst explanations

In [ ]:
# ================================# ------------------------------------------------
def _ensure_loaded():
    return isinstance(uploaded_df, pd.DataFrame) and not uploaded_df.empty

def _services():
    if not _ensure_loaded() or "service" not in uploaded_df.columns:
        return []
    return sorted(uploaded_df["service"].dropna().unique().tolist())

def _error_types():
    if not _ensure_loaded() or "error_type" not in uploaded_df.columns:
        return []
    return sorted(uploaded_df["error_type"].dropna().unique().tolist())

def _df_and_markdown(result, max_rows=15):
    if isinstance(result, str):
        return pd.DataFrame(), result
    if result is None or (hasattr(result, "empty") and result.empty):
        return pd.DataFrame(), "✅ No results."
    md = result.head(max_rows).to_markdown(index=False)
    return result, md

# ------------------------------------------------
# Safe query handlers
# ------------------------------------------------
def ui_error_summary():
    return _df_and_markdown(get_error_summary())

def ui_top_errors(limit):
    return _df_and_markdown(get_top_errors(int(limit)))

def ui_errors_by_service(service):
    if not service:
        return pd.DataFrame(), "Pick a service first."
    return _df_and_markdown(get_errors_by_service(service))

def ui_errors_in_range(start_date, end_date):
    if not start_date or not end_date:
        return pd.DataFrame(), "Pick both start and end dates."
    return _df_and_markdown(get_errors_in_date_range(start_date, end_date))

# ------------------------------------------------
# LLM (explain only — no querying)
# ------------------------------------------------
def ui_explain_error_type(error_type):
    if not error_type:
        return "Pick an error type first."

    base = ""
    try:
        base = explain_error_type(error_type)
    except Exception:
        pass

    prompt = (
        "Explain this software error type in plain English for a non‑technical analyst.\n"
        "Limit to 2–3 sentences and suggest one next check.\n\n"
        f"Error type: {error_type}\n"
        f"Known hint: {base}"
    )

    # If Cell 6 wasn't run, llm_invoke won't exist — fall back to the baseline text
    if "llm_invoke" not in globals():
        return base or "(LLM not initialized. Run Cell 6 to enable explanations.)"

    try:
        return llm_invoke(prompt, max_new_tokens=120)
    except Exception as e:
        return f"{base}\n\n(LLM unavailable: {e})"

def ui_narrate_table(table):
    if not isinstance(table, pd.DataFrame) or table.empty:
        return "Run a query first."

    if "llm_invoke" not in globals():
        return "(LLM not initialized. Run Cell 6 to enable narration.)"

    csv_text = table.head(25).to_csv(index=False)
    prompt = (
        "Summarize the table below for a non‑technical analyst.\n"
        "Call out the biggest pattern and one recommended next step.\n\n"
        f"{csv_text}"
    )

    try:
        return llm_invoke(prompt, max_new_tokens=200)
    except Exception as e:
        return f"(LLM unavailable: {e})"

# ------------------------------------------------
# Simple dropdown refresh functions (less error-prone)
# ------------------------------------------------
def refresh_services_only():
    return gr.update(choices=_services())

def refresh_error_types_only():
    return gr.update(choices=_error_types())

# ------------------------------------------------
# Build ONE UI (Upload + Queries + Explain)
# ------------------------------------------------
with gr.Blocks(title="Analyst‑Safe Log Explorer") as demo:
    gr.Markdown(
        "# Analyst‑Safe Log Explorer\n"
        "Guided log analysis for analysts. **No raw querying, no Python, no SQL.**\n\n"
        "✅ The model never runs code — it only explains patterns we already computed."
    )

    # ----------------------------
    # Upload / Profile (in the SAME app)
    # ----------------------------
    with gr.Accordion("1) Upload logs", open=True):
        with gr.Row():
            file_in = gr.File(label="Upload logs (.csv, .jsonl, .xlsx)")
            profile_btn = gr.Button("Show dataset info")

        preview = gr.Dataframe(label="Preview (first rows)", interactive=False)
        info = gr.Markdown()

        file_in.upload(load_logs_gradio, inputs=file_in, outputs=preview)
        profile_btn.click(dataset_profile, outputs=info)

    # Shared result table for both tabs (so narrate works)
    result_table = gr.Dataframe(label="Results", interactive=False)
    result_md = gr.Markdown()

    # ----------------------------
    # Safe Queries
    # ----------------------------
    with gr.Tab("Safe Queries"):
        with gr.Row():
            btn_summary = gr.Button("Error Summary", variant="primary")
            btn_top = gr.Button("Top Errors")
            top_k = gr.Slider(3, 20, value=5, step=1, label="Top K")

        with gr.Row():
            service_dd = gr.Dropdown(label="Service", choices=[], allow_custom_value=True)
            refresh_services_btn = gr.Button("🔄 Refresh Services")
            btn_service = gr.Button("Errors by Service")

        with gr.Row():
            start = gr.Textbox(label="Start date (YYYY‑MM‑DD)")
            end = gr.Textbox(label="End date (YYYY‑MM‑DD)")
            btn_range = gr.Button("Errors in Date Range")

        btn_summary.click(ui_error_summary, outputs=[result_table, result_md])
        btn_top.click(ui_top_errors, inputs=top_k, outputs=[result_table, result_md])
        btn_service.click(ui_errors_by_service, inputs=service_dd, outputs=[result_table, result_md])
        btn_range.click(ui_errors_in_range, inputs=[start, end], outputs=[result_table, result_md])

        refresh_services_btn.click(refresh_services_only, outputs=service_dd)

        # Auto-refresh service dropdown after upload
        file_in.upload(lambda f: refresh_services_only(), inputs=file_in, outputs=service_dd)

    # ----------------------------
    # Explain (LLM)
    # ----------------------------
    with gr.Tab("Explain (LLM)"):
        gr.Markdown("### Plain‑English explanations (LLM does not query data)")

        with gr.Row():
            err_dd = gr.Dropdown(label="Error Type", choices=[], allow_custom_value=True)
            refresh_err_btn = gr.Button("🔄 Refresh Error Types")

        explain_btn = gr.Button("Explain Error Type", variant="primary")
        explain_out = gr.Markdown()

        explain_btn.click(ui_explain_error_type, inputs=err_dd, outputs=explain_out)
        refresh_err_btn.click(refresh_error_types_only, outputs=err_dd)

        # Auto-refresh error type dropdown after upload
        file_in.upload(lambda f: refresh_error_types_only(), inputs=file_in, outputs=err_dd)

        gr.Markdown("### Narrate last results table")
        narrate_btn = gr.Button("Narrate Results")
        narrate_out = gr.Markdown()
        narrate_btn.click(ui_narrate_table, inputs=result_table, outputs=narrate_out)

demo.launch(share=True)

# COLAB CELL 7 — Analyst‑Safe Log Explorer (Gradio) — SINGLE APP
# ================================

!pip -q install -U gradio

import gradio as gr
import pandas as pd

# ------------------------------------------------
# Ensure uploaded_df always exists
# ------------------------------------------------
try:
    uploaded_df
except NameError:
    uploaded_df = None

# ------------------------------------------------
# Helpers


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://6bc56a4a5d3d2791b4.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
